# Spam-Klassifikation — Von TF-IDF bis zum Neuronalen Netz

Dieses Notebook vergleicht verschiedene Ansätze zur Spam-Erkennung:

1. **Datenexploration** — SMS Spam Collection laden und analysieren
2. **Feature Engineering** — TF-IDF-Vektorisierung + Metafeatures
3. **Modellvergleich** — Naive Bayes vs. Logistic Regression
4. **Fehleranalyse** — False Positives & False Negatives
5. **Interpretation** — Wichtigste Spam-Wörter

Basiert auf `spam_classifier.py`. Dataset: [SMS Spam Collection (UCI)](https://archive.ics.uci.edu/dataset/228/sms+spam+collection).

## 1. Importe und Setup

In [ ]:
import os
import shutil
import time
import zipfile
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

print("✅ Alle Importe erfolgreich!")

## 2. Daten laden: SMS Spam Collection

Der Datensatz enthält 5.574 englische SMS-Nachrichten, davon 747 Spam (13,4%).
Jede Nachricht ist mit `ham` (kein Spam) oder `spam` gelabelt.

In [ ]:
def load_spam_data():
    """Lädt das SMS Spam Collection Dataset."""
    cache_dir = os.path.join(os.path.dirname(__file__) or ".", ".data_cache")
    os.makedirs(cache_dir, exist_ok=True)

    zip_path = os.path.join(cache_dir, "smsspamcollection.zip")
    extract_path = os.path.join(cache_dir, "smsspamcollection")

    if not os.path.exists(extract_path):
        url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
        print("  Lade SMS Spam Collection...")
        with urlopen(url, timeout=30) as response, open(zip_path, "wb") as f:
            shutil.copyfileobj(response, f)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_path)

    for f in os.listdir(extract_path):
        if f.endswith((".csv", ".txt")) or "SMSSpamCollection" in f:
            path = os.path.join(extract_path, f)
            break
    else:
        raise FileNotFoundError("Keine Datendatei gefunden")

    df = pd.read_csv(path, sep="\t", header=None, names=["label", "message"])
    df["label"] = (df["label"] == "spam").astype(int)
    return df


print("📦 Lade SMS Spam Collection...")
df = load_spam_data()
print(f"   {len(df):,} Nachrichten ({df['label'].sum():,} Spam, "
      f"{len(df) - df['label'].sum():,} Ham)")
print(f"   Spam-Anteil: {df['label'].mean()*100:.1f}%")

## 3. Datenexploration

In [ ]:
# Klassenverteilung
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df["label"].value_counts()
axes[0].bar(["Ham (kein Spam)", "Spam"], [counts[0], counts[1]],
           color=["steelblue", "coral"], edgecolor="white")
axes[0].set_title("Klassenverteilung")
axes[0].set_ylabel("Anzahl")
for i, (label, count) in enumerate(zip(["Ham", "Spam"], [counts[0], counts[1]])):
    axes[0].text(i, count + 20, f"{count:,}", ha="center", fontsize=11)

# Nachrichtenlänge pro Klasse
df["length"] = df["message"].str.len()
ham_lengths = df[df["label"] == 0]["length"]
spam_lengths = df[df["label"] == 1]["length"]
axes[1].hist(ham_lengths, bins=50, alpha=0.6, label="Ham", color="steelblue")
axes[1].hist(spam_lengths, bins=50, alpha=0.6, label="Spam", color="coral")
axes[1].set_title("Nachrichtenlänge nach Klasse")
axes[1].set_xlabel("Zeichen")
axes[1].set_ylabel("Anzahl")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"📏 Durchschnittliche Länge:")
print(f"   Ham:  {ham_lengths.mean():.0f} Zeichen")
print(f"   Spam: {spam_lengths.mean():.0f} Zeichen")

In [ ]:
# Beispiel-Nachrichten
print("📨 Beispiel-Nachrichten:\n")
print("── Ham ──")
for msg in df[df["label"] == 0]["message"].sample(3, random_state=42):
    print(f"  • {msg[:100]}")
print("\n── Spam ──")
for msg in df[df["label"] == 1]["message"].sample(3, random_state=42):
    print(f"  • {msg[:100]}")

## 4. Feature Engineering

Wir extrahieren zwei Arten von Features:

### 4.1 TF-IDF-Vektorisierung
- **TF (Term Frequency)**: Wie oft kommt ein Wort in einer Nachricht vor?
- **IDF (Inverse Document Frequency)**: Wie selten ist das Wort im gesamten Korpus?
- **TF-IDF** = TF × IDF — gewichtet häufige, aber nicht zu allgemeine Wörter
- Parameter: max. 5.000 Features, Unigrams + Bigrams, englische Stopwörter entfernt

### 4.2 Metafeatures
Zusätzliche numerische Merkmale:
- Nachrichtenlänge, Wortanzahl
- Anzahl Großbuchstaben, Ziffern, Ausrufezeichen
- Enthält URL? Enthält Telefonnummer?

In [ ]:
def create_features(df):
    """Erstellt TF-IDF-Vektoren + Metafeatures."""
    vectorizer = TfidfVectorizer(
        max_features=5000,
        stop_words="english",
        ngram_range=(1, 2),
    )
    X_tfidf = vectorizer.fit_transform(df["message"])

    meta = pd.DataFrame({
        "length": df["message"].str.len(),
        "num_words": df["message"].str.split().str.len(),
        "num_caps": df["message"].str.count(r"[A-Z]"),
        "num_digits": df["message"].str.count(r"\d"),
        "num_exclamations": df["message"].str.count(r"!"),
        "has_url": df["message"].str.contains(r"http|www\.", regex=True).astype(int),
        "has_phone": df["message"].str.contains(r"\d{3,}").astype(int),
    }).values

    X_combined = hstack([X_tfidf, csr_matrix(meta)])
    return X_combined, vectorizer


print("🔧 Extrahiere Features (TF-IDF + Metafeatures)...")
X, vectorizer = create_features(df)
y = df["label"].values
print(f"   Feature-Dimension: {X.shape[1]:,}")
print(f"   Davon TF-IDF: 5.000 | Metafeatures: 7")

## 5. Train/Test-Split

80% Training, 20% Test — stratifiziert nach Klassenverteilung.

In [ ]:
X_train, X_test, y_train, y_test, _train_idx, test_idx = train_test_split(
    X, y, np.arange(len(y)), test_size=0.2, random_state=42, stratify=y
)
print(f"   Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"   Spam-Anteil Train: {y_train.mean()*100:.1f}% | Test: {y_test.mean()*100:.1f}%")

## 6. Modellvergleich

Wir vergleichen zwei klassische ML-Modelle:

| Modell | Beschreibung |
|--------|-------------|
| **Multinomial Naive Bayes** | Wahrscheinlichkeitsbasiert, schnell, gut für Text |
| **Logistic Regression** | Lineares Modell, interpretierbar, oft stärker |

Metriken: **Accuracy, Precision, Recall, F1-Score**

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    """Trainiert und evaluiert ein Modell."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    return {
        "name": name,
        "model": model,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "predictions": y_pred,
    }


print("🤖 Trainiere Modelle...\n")
models = [
    ("Naive Bayes", MultinomialNB(alpha=0.1)),
    ("Logistic Regression", LogisticRegression(max_iter=1000, C=1.0)),
]

results = []
for name, model in models:
    print(f"   ⏳ {name}...")
    t0 = time.time()
    res = evaluate_model(name, model, X_train, X_test, y_train, y_test)
    train_time = time.time() - t0
    res["train_time"] = train_time
    results.append(res)
    print(f"   ✅ {name}: F1={res['f1']:.4f} (in {train_time:.2f}s)")

## 7. Ergebnisse

In [ ]:
# Vergleichstabelle
print("📊 Ergebnisse:\n")
print(f"   {'Modell':<25s} {'Acc':>6s} {'Prec':>6s} {'Rec':>6s} {'F1':>6s} {'Zeit':>7s}")
print("   " + "-" * 62)
for r in results:
    print(f"   {r['name']:<25s} "
          f"{r['accuracy']:6.3f} {r['precision']:6.3f} "
          f"{r['recall']:6.3f} {r['f1']:6.3f} "
          f"{r['train_time']:6.2f}s")

# Bestes Modell
best = max(results, key=lambda r: r["f1"])
print(f"\n🏆 Bestes Modell: {best['name']} (F1={best['f1']:.3f})")

In [ ]:
# Visualisierung der Ergebnisse
fig, ax = plt.subplots(figsize=(10, 5))
metrics = ["accuracy", "precision", "recall", "f1"]
x = np.arange(len(metrics))
width = 0.35

for i, r in enumerate(results):
    values = [r[m] for m in metrics]
    bars = ax.bar(x + i * width, values, width, label=r["name"],
                  color=["steelblue", "coral"][i], edgecolor="white")
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", fontsize=8)

ax.set_ylabel("Wert")
ax.set_title("Modellvergleich: Metriken im Überblick")
ax.set_xticks(x + width / 2)
ax.set_xticklabels(["Accuracy", "Precision", "Recall", "F1"])
ax.legend()
ax.set_ylim(0, 1.15)
plt.tight_layout()
plt.show()

## 8. Fehleranalyse

Wir analysieren die Fehler des besten Modells:
- **False Positives (FP)**: Ham-Nachrichten, die fälschlich als Spam markiert wurden
- **False Negatives (FN)**: Spam-Nachrichten, die durchgerutscht sind

In [ ]:
y_pred = best["predictions"]
errors = np.where(y_test != y_pred)[0]
print(f"🔍 Fehleranalyse ({best['name']}):")
print(f"   {len(errors)}/{len(y_test)} Fehler insgesamt ({len(errors)/len(y_test)*100:.1f}%)")

# False Positives
fp = errors[y_test[errors] == 0]
print(f"\n   ❌ False Positives (Ham → Spam): {len(fp)}")
for i in fp[:5]:
    orig_idx = test_idx[i]
    print(f"   • \"{df.iloc[orig_idx]['message'][:100]}...\"")

# False Negatives
fn = errors[y_test[errors] == 1]
print(f"\n   ⚠️  False Negatives (Spam → Ham): {len(fn)}")
for i in fn[:5]:
    orig_idx = test_idx[i]
    print(f"   • \"{df.iloc[orig_idx]['message'][:100]}...\"")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Ham", "Spam"],
            yticklabels=["Ham", "Spam"])
ax.set_xlabel("Vorhergesagt")
ax.set_ylabel("Tatsächlich")
ax.set_title(f"Confusion Matrix — {best['name']}")
plt.tight_layout()
plt.show()

## 9. Interpretation: Wichtigste Spam-Wörter

Bei Logistic Regression können wir die Koeffizienten direkt interpretieren:
Je höher der Koeffizient, desto stärker weist das Wort auf Spam hin.

In [ ]:
# Top-Spam-Wörter aus Logistic Regression
lr_result = next((r for r in results if r["name"] == "Logistic Regression"), None)
if lr_result:
    coef = lr_result["model"].coef_[0]
    n_tfidf = len(vectorizer.get_feature_names_out())
    feature_names = vectorizer.get_feature_names_out()

    # Top-15 Spam-Wörter
    top_idx = np.argsort(coef[:n_tfidf])[-15:][::-1]

    print("📝 Top-15 Spam-Indikatoren (Logistic Regression):\n")
    fig, ax = plt.subplots(figsize=(10, 5))
    words = [feature_names[i] for i in top_idx]
    weights = [coef[i] for i in top_idx]
    colors = ["coral" if w > 0 else "steelblue" for w in weights]
    ax.barh(range(len(words)), weights, color=colors, edgecolor="white")
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words)
    ax.invert_yaxis()
    ax.set_xlabel("Koeffizient (positiv = Spam)")
    ax.set_title("Wichtigste Spam-Indikatoren")
    plt.tight_layout()
    plt.show()

    # Top-15 Ham-Wörter
    bottom_idx = np.argsort(coef[:n_tfidf])[:15]
    print("\n📝 Top-15 Ham-Indikatoren:\n")
    for i, idx in enumerate(bottom_idx, 1):
        print(f"   {i:2d}. {feature_names[idx]:25s} ({coef[idx]:.3f})")

## 10. Eigene Nachrichten testen

Teste das beste Modell mit eigenen SMS-Nachrichten!

In [ ]:
def predict_message(message, model, vectorizer):
    """Sagt vorher, ob eine Nachricht Spam ist."""
    # Gleiche Metafeatures wie beim Training
    import re as re_module
    meta = np.array([[
        len(message),
        len(message.split()),
        len(re_module.findall(r"[A-Z]", message)),
        len(re_module.findall(r"\d", message)),
        message.count("!"),
        int(bool(re_module.search(r"http|www\.", message))),
        int(bool(re_module.search(r"\d{3,}", message))),
    ]])
    X_tfidf = vectorizer.transform([message])
    X_combined = hstack([X_tfidf, csr_matrix(meta)])
    proba = model.predict_proba(X_combined)[0]
    pred = model.predict(X_combined)[0]
    return "SPAM" if pred == 1 else "HAM", proba[1]


# Teste eigene Nachrichten
test_messages = [
    "Hey, are we still meeting for lunch today?",
    "CONGRATULATIONS! You've won a FREE iPhone! Click here to claim: http://spam.com",
    "Don't forget to bring the documents tomorrow. Thanks!",
    "URGENT: Your account has been compromised. Call 0800-123-456 immediately!",
]

best_model = best["model"]
print("🧪 Eigene Nachrichten testen:\n")
for msg in test_messages:
    label, proba = predict_message(msg, best_model, vectorizer)
    emoji = "🔴" if label == "SPAM" else "🟢"
    print(f"{emoji} {label} ({proba*100:.1f}% Spam): \"{msg[:80]}...\"")

## Zusammenfassung

In diesem Notebook haben wir:

1. ✅ Die **SMS Spam Collection** geladen und exploriert (5.574 Nachrichten, 13,4% Spam)
2. ✅ **TF-IDF-Features** (5.000 Unigram+Bigram) + **7 Metafeatures** extrahiert
3. ✅ **Naive Bayes** und **Logistic Regression** trainiert und verglichen
4. ✅ **Fehleranalyse** mit False Positives und False Negatives durchgeführt
5. ✅ Die **wichtigsten Spam-Indikatoren** identifiziert
6. ✅ Das Modell mit **eigenen Nachrichten** getestet

**Nächste Schritte**:
- Neuronales Netz mit Word Embeddings (z. B. via spaCy oder FastText)
- Cross-Validation für robustere Ergebnisse
- Hyperparameter-Tuning mit GridSearchCV
- Weitere Features: Absender-Info, Nachrichtenstruktur